# Pandas Basics — Example 01

NumPy gave us a fast array. Pandas builds two things on top of it:

- **Series** — one column of data with a label on every row
- **DataFrame** — many Series side by side, i.e. a table

Backend analogy: a `DataFrame` is a `List<Row>` that already knows its column
names and column types, plus it has SQL built in — select, filter, join, group
by. A `Series` is one column of that table.

This notebook walks through:

1. Series — one labelled column
2. DataFrame — a table
3. Reading a real CSV file
4. Renaming columns, and the `inplace` trap
5. Picking columns and rows
6. Filtering rows with a condition
7. Missing values — find them and fill them
8. Dropping a column
9. Stacking frames — `concat` (the old `append` is gone)
10. Merging frames — a SQL join

The two imports at the top of almost every pandas file. `pd` and `np` are the
standard short names — everybody uses them.

`np.random.seed(42)` fixes the random numbers. Without it every re-run of this
notebook prints different values and the notes below stop matching the output.

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

## 1. Series — one column with an index

A Series is a single column. Two parts go with it:

- the **values** — the actual data
- the **index** — a label for each row

If you do not give an index, pandas makes one for you: 0, 1, 2, 3, ...

In [2]:
s1 = pd.Series(data=[10,20,30,40,50])
s1

0    10
1    20
2    30
3    40
4    50
dtype: int64

In [3]:
s2 = pd.Series(data=[100,200,300,400,500])
s2

0    100
1    200
2    300
3    400
4    500
dtype: int64

Look at the print-out. The left column is the index (0 to 4), the right column
is the data. At the bottom `dtype: int64` says every value is a 64-bit integer.

One dtype for the whole Series — same rule as a NumPy array.

In [4]:
s3 = pd.Series(data=[1000,2000,3000,4000,5000],index=list('vwxyz'))
s3

v    1000
w    2000
x    3000
y    4000
z    5000
dtype: int64

Here the index is our own: `v w x y z`. `list('vwxyz')` is a quick trick to
turn a string into a list of single characters.

The index is not just decoration. It is the key pandas uses when it lines up
two objects — see the very next cell.

## 2. DataFrame — many Series side by side

A DataFrame is a table. Give it a dictionary and each key becomes a column
name, each value becomes that column's data.

In [5]:
df = pd.DataFrame({'C1':s1,"C2":s2})
df

,C1,C2
0,10,100
1,20,200
2,30,300
3,40,400
4,50,500


`s1` and `s2` both had the index 0–4, so the rows lined up perfectly.

This matching is by **index label**, not by position. If `s1` had the index
`v w x y z` instead, pandas would not find those labels in `s2` and the result
would be full of `NaN` (missing). Alignment is automatic and it is the number
one surprise for people coming from plain arrays.

In [6]:
df = pd.DataFrame(data=np.random.randint(1,100,size=(1000,5)),columns=list('abcde'))
df


,a,b,c,d,e
0,52,93,15,72,61
1,21,83,87,75,75
2,88,24,3,22,53
3,2,88,30,38,2
4,64,60,21,33,76
...,...,...,...,...,...
995,13,32,46,23,45
996,17,85,66,6,18
997,94,22,33,30,81
998,32,42,78,10,36


A DataFrame can also be built straight from a NumPy array.

- `np.random.randint(1,100,size=(1000,5))` → 1000 rows, 5 columns of random
  integers from 1 to 99
- `columns=list('abcde')` → names those five columns `a b c d e`

Notice pandas does not print all 1000 rows. It shows the first 5, then `..`,
then the last 5, and tells you the real size at the bottom:
`[1000 rows x 5 columns]`.

## 3. Read a real file

`pd.read_csv()` is the door into pandas. It reads the file, guesses the column
names from the header row, and guesses each column's dtype.

This dataset is small: age, income, and whether the person bought a car.

In [7]:
dataset = pd.read_csv('/Users/kaliprasad/Documents/MACHINE_LEARNING/AI_ML_Series/02_pandas/DataSet/Car.csv')
dataset

,Age,Income,Car
0,28,37000,0
1,27,88000,0
2,28,59000,0
3,32,86000,0
4,33,149000,1
...,...,...,...
95,28,89000,0
96,34,43000,0
97,30,79000,0
98,20,36000,0


In [8]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Age     100 non-null    int64
 1   Income  100 non-null    int64
 2   Car     100 non-null    int64
dtypes: int64(3)
memory usage: 2.5 KB


`.info()` is the first command to run on any new dataset. It answers three
questions at once:

| What it shows | Why it matters |
|---|---|
| `RangeIndex: 100 entries` | how many rows you have |
| `Non-Null Count` | how many values are **missing** in each column |
| `Dtype` | is the column a number or text? |

Here all three columns say `100 non-null` out of 100 rows — nothing is
missing. And all three are `int64`, so all three are numbers.

That "non-null" line is the fastest missing-value check there is.

In [9]:
dataset.head()

,Age,Income,Car
0,28,37000,0
1,27,88000,0
2,28,59000,0
3,32,86000,0
4,33,149000,1


In [10]:
dataset.tail()

,Age,Income,Car
95,28,89000,0
96,34,43000,0
97,30,79000,0
98,20,36000,0
99,26,80000,0


`.head()` shows the first 5 rows, `.tail()` shows the last 5. Pass a number
for more — `dataset.head(20)`.

Use them to eyeball the data. A quick look at the top and the bottom catches
things like a stray total row at the end of a spreadsheet export.

In [11]:
dataset.Car.value_counts()

Car
0    93
1     7
Name: count, dtype: int64

`.value_counts()` counts how many times each distinct value appears. On the
`Car` column it acts like `GROUP BY Car, COUNT(*)`.

Result: 93 people did not buy, 7 did.

That is worth stopping on. This is an **imbalanced** target — one class is 13
times bigger than the other. A model that always guesses "0" would be 93%
accurate and completely useless. Remember this when we reach classification.

Small note on style: `dataset.Car` and `dataset['Car']` do the same thing. The
dot form only works when the column name has no space and does not clash with
a real method name, so `dataset['Car']` is the safer habit.

## 4. Rename columns — and the `inplace` trap

This is the single most common beginner confusion in pandas, so go slowly.

Most pandas methods **do not change the original**. They build a changed copy
and hand it back. If you do not catch that copy in a variable, the work is
thrown away.

In [12]:
dataset.rename(columns={'Car':'Target'})


,Age,Income,Target
0,28,37000,0
1,27,88000,0
2,28,59000,0
3,32,86000,0
4,33,149000,1
...,...,...,...
95,28,89000,0
96,34,43000,0
97,30,79000,0
98,20,36000,0


In [13]:
dataset

,Age,Income,Car
0,28,37000,0
1,27,88000,0
2,28,59000,0
3,32,86000,0
4,33,149000,1
...,...,...,...
95,28,89000,0
96,34,43000,0
97,30,79000,0
98,20,36000,0


Read those two outputs together.

The first cell printed a table with a `Target` column — so the rename did
happen. The second cell printed `dataset` and the column is still called
`Car`.

Nothing was saved. `rename()` returned a **new** DataFrame, the notebook
displayed it, and then it was dropped.

Two ways to actually keep the change:

- `dataset = dataset.rename(columns=...)` — reassign (preferred today)
- `dataset.rename(columns=..., inplace=True)` — edit in place

In [14]:
dataset.rename(columns={'Car':'Target'},inplace=True)
dataset

,Age,Income,Target
0,28,37000,0
1,27,88000,0
2,28,59000,0
3,32,86000,0
4,33,149000,1
...,...,...,...
95,28,89000,0
96,34,43000,0
97,30,79000,0
98,20,36000,0


Now it stuck. `inplace=True` means "change this object, return nothing".

A word of warning: `inplace=True` is on its way out of pandas. It saves no
memory, it does not work in a method chain, and the pandas team discourages it.
Prefer the reassign form `df = df.rename(...)` in real code.

In [15]:
dataset.columns

Index(['Age', 'Income', 'Target'], dtype='str')

`.columns` gives back the column names as an `Index` object. Handy for a quick
check after a rename, or for looping over columns.

## 5. Pick columns and rows

Square brackets pick columns. The number of brackets decides what you get
back.

In [16]:
dataset[['Age','Target']]

,Age,Target
0,28,0
1,27,0
2,28,0
3,32,0
4,33,1
...,...,...
95,28,0
96,34,0
97,30,0
98,20,0


In [17]:
dataset['Target']

0     0
1     0
2     0
3     0
4     1
     ..
95    0
96    0
97    0
98    0
99    0
Name: Target, Length: 100, dtype: int64

Compare the two outputs carefully:

| Code | Result | Why |
|---|---|---|
| `dataset[['Age','Target']]` | a **DataFrame** | the inner `[...]` is a *list* of columns |
| `dataset['Target']` | a **Series** | one name, so one column |

The double bracket is not special syntax. It is a list inside the normal
bracket. Ask for a list of columns, get a table back; ask for one name, get a
single column back.

This matters later: scikit-learn wants `X` as a 2-D table and `y` as a 1-D
column, which is exactly this difference.

In [18]:
dataset.loc[50]

Age          20
Income    74000
Target        0
Name: 50, dtype: int64

In [19]:
dataset.loc[50:60]

,Age,Income,Target
50,20,74000,0
51,26,15000,0
52,41,45000,0
53,31,76000,0
54,36,50000,0
55,40,47000,0
56,31,15000,0
57,46,59000,0
58,29,75000,0
59,26,30000,0


`.loc[]` selects **rows by label**.

`dataset.loc[50]` gives one row back as a Series — the column names become the
index of that Series.

`dataset.loc[50:60]` gives rows 50 to 60. Count them: that is **11 rows, 60
included**. This is the one place pandas breaks the Python rule. A normal
Python slice `[50:60]` stops before 60, but `.loc` slices by label and includes
the end label.

Related: `.iloc[]` selects by **position** instead of label, and `.iloc[50:60]`
does stop before 60. With this default 0,1,2... index the two look identical,
which is exactly why the difference bites people later, once the index is dates
or IDs.

## 6. Filter rows with a condition

A comparison on a column does not return one True/False. It returns a whole
column of True/False, one per row — a **boolean mask**. Feed that mask back
into the brackets and pandas keeps only the `True` rows.

Think of it as the `WHERE` clause of SQL.

In [20]:
(dataset['Income'] > 100000).head()

0    False
1    False
2    False
3    False
4     True
Name: Income, dtype: bool

In [21]:
dataset[(dataset['Income']>10000)]

,Age,Income,Target
0,28,37000,0
1,27,88000,0
2,28,59000,0
3,32,86000,0
4,33,149000,1
...,...,...,...
95,28,89000,0
96,34,43000,0
97,30,79000,0
98,20,36000,0


The first cell shows the mask itself — plain True/False values.

The second cell uses a mask as a filter, but look at the row count:
`[100 rows x 3 columns]`. Nothing was filtered out, because every income in
this file is above 10,000. The code is right; the threshold was just too low to
remove anything. Always check the row count after a filter.

In [22]:
dataset[dataset['Income'] > 100000]

,Age,Income,Target
4,33,149000,1
38,30,107000,1
60,32,135000,1
69,29,148000,1
72,34,115000,0
73,26,118000,0
83,32,117000,1


In [23]:
dataset[(dataset['Income'] > 100000) & (dataset['Age'] < 32)]

,Age,Income,Target
38,30,107000,1
69,29,148000,1
73,26,118000,0


A useful threshold this time — 7 high earners instead of all 100 rows. Adding
the second condition (`Age < 32`) narrows it further, down to 3 rows.

For two conditions use `&` (and) and `|` (or), **not** the Python words `and` /
`or`. And each condition needs its own brackets, because `&` binds tighter than
`>` in Python. Missing brackets here is a classic error message.

## 7. Missing values — find them and fill them

Real files have holes. Pandas shows a hole as `NaN` ("not a number").

One thing to know up front: a column with even one `NaN` in it becomes a
`float` column, because `NaN` is a float value. That is why prices below print
as `70.0` and not `70`.

In [24]:
example = pd.read_csv('/Users/kaliprasad/Documents/MACHINE_LEARNING/AI_ML_Series/02_pandas/DataSet/abc.csv')
example

,iteams,price
0,A,70.0
1,B,NaN
2,C,50.0
3,D,NaN
4,E,40.0
5,F,32.0
6,G,45.0
7,H,69.0
8,I,NaN
9,J,NaN


In [25]:
example['price'].notnull()

0     True
1    False
2     True
3    False
4     True
5     True
6     True
7     True
8    False
9    False
Name: price, dtype: bool

In [26]:
example[example['price'].isnull()]

,iteams,price
1,B,NaN
3,D,NaN
8,I,NaN
9,J,NaN


Two mirror-image checks:

- `.isnull()` → `True` where the value **is** missing
- `.notnull()` → `True` where the value **is present**

On their own they just print True/False. Put one inside brackets, as in the
last cell, and you get the actual rows that are missing — rows B, D, I and J.

For a fast count instead of a list, use `example.isnull().sum()`.

In [27]:
example['price'].fillna(example['price'].max())

0    70.0
1    70.0
2    50.0
3    70.0
4    40.0
5    32.0
6    45.0
7    69.0
8    70.0
9    70.0
Name: price, dtype: float64

In [28]:
example

,iteams,price
0,A,70.0
1,B,NaN
2,C,50.0
3,D,NaN
4,E,40.0
5,F,32.0
6,G,45.0
7,H,69.0
8,I,NaN
9,J,NaN


`fillna()` replaces every `NaN` with the value you give it — here the maximum
price, 70.

But look at the second cell: `example` still has its `NaN` values. Same trap as
`rename()` in section 4. `fillna()` returned a filled **copy**, and we never
stored it.

To keep the result, assign it back.

In [29]:
example['price'] = example['price'].fillna(example['price'].std())
example

,iteams,price
0,A,70.000000
1,B,15.517732
2,C,50.000000
3,D,15.517732
4,E,40.000000
5,F,32.000000
6,G,45.000000
7,H,69.000000
8,I,15.517732
9,J,15.517732


This one was assigned back with `example['price'] = ...`, so the change is
permanent. The four holes are now 15.517732 — the standard deviation of the six
prices that were present.

**Careful here.** `example` now has zero `NaN` values left. Any `fillna()` you
run after this line has nothing to fill and will look like it did nothing. That
is why the two cells below re-read the CSV first — each fill strategy needs a
fresh copy of the data with the holes back in it.

In [30]:
example = pd.read_csv('/Users/kaliprasad/Documents/MACHINE_LEARNING/AI_ML_Series/02_pandas/DataSet/abc.csv')
example['price'].fillna(example['price'].min())

0    70.0
1    32.0
2    50.0
3    32.0
4    40.0
5    32.0
6    45.0
7    69.0
8    32.0
9    32.0
Name: price, dtype: float64

In [31]:
example = pd.read_csv('/Users/kaliprasad/Documents/MACHINE_LEARNING/AI_ML_Series/02_pandas/DataSet/abc.csv')
example['price'].fillna(example['price'].max())

0    70.0
1    70.0
2    50.0
3    70.0
4    40.0
5    32.0
6    45.0
7    69.0
8    70.0
9    70.0
Name: price, dtype: float64

Now the difference is visible: the first fills the holes with 32.0 (the
smallest price), the second with 70.0 (the largest).

Which value should you actually use? A quick guide:

| Fill with | When |
|---|---|
| **mean** | numbers, roughly symmetric, no big outliers |
| **median** | numbers with outliers — the safe default |
| **mode** | text / category columns |
| a fixed value like 0 or `"Unknown"` | when "missing" is itself meaningful |
| drop the rows (`dropna()`) | very few rows affected |

`std` — used above — is not a real strategy. It is the *spread* of the data,
not a typical value, so it drops a meaningless number into the column. It is
fine as syntax practice, but do not carry it into a real project.

Bigger point: whatever you choose, the fill value must be learned from the
**training data only** and then applied to the test data. Computing it over the
whole file leaks information about the test set into training. That is what
scikit-learn's `SimpleImputer` is for, and it is coming later.

## 8. Drop a column

`axis='columns'` (or `axis=1`) means "drop by column name". `axis='index'`
(or `axis=0`, the default) means "drop by row label".

In [32]:
df.drop(['b'],axis='columns')

,a,c,d,e
0,52,15,72,61
1,21,87,75,75
2,88,3,22,53
3,2,30,38,2
4,64,21,33,76
...,...,...,...,...
995,13,46,23,45
996,17,66,6,18
997,94,33,30,81
998,32,78,10,36


In [33]:
df

,a,b,c,d,e
0,52,93,15,72,61
1,21,83,87,75,75
2,88,24,3,22,53
3,2,88,30,38,2
4,64,60,21,33,76
...,...,...,...,...,...
995,13,32,46,23,45
996,17,85,66,6,18
997,94,22,33,30,81
998,32,42,78,10,36


Third time seeing this pattern, so it should feel familiar now: the drop
printed a table without column `b`, but `df` itself still has `b`. A copy was
returned and thrown away.

In [34]:
df.drop(['b'],axis='columns',inplace=True)
df

,a,c,d,e
0,52,15,72,61
1,21,87,75,75
2,88,3,22,53
3,2,30,38,2
4,64,21,33,76
...,...,...,...,...
995,13,46,23,45
996,17,66,6,18
997,94,33,30,81
998,32,78,10,36


With `inplace=True` the column is really gone from `df`.

**The rule for the whole library:** unless you pass `inplace=True` or assign
the result back, a pandas method leaves your data untouched. When something
"did not work", check this first — it is the cause more often than not.

## 9. Stack frames — `concat` replaces `append`

Two frames, different column names.

In [35]:
data1 = pd.DataFrame(data=np.random.randint(1,100,size=(100,4)),columns=list('abcd'))
data2 = pd.DataFrame(data=np.random.randint(1,100,size=(100,3)),columns=list('xyz'))

In [36]:
data1,data2

(     a   b   c   d
 0   39  47  19  15
 1   28  24  72   9
 2   47  33  62  97
 3   19  67  74  40
 4   47  65  29  50
 ..  ..  ..  ..  ..
 95  33  59  43  79
 96  12  40  20  94
 97  38   3  28  56
 98  90  94  13  77
 99  66  37  49  66
 
 [100 rows x 4 columns],
      x   y   z
 0   36  54  26
 1   75  72  40
 2   43  22  20
 3    8  29  40
 4   90  30  81
 ..  ..  ..  ..
 95  52   9  54
 96   1  44  99
 97  75  98  80
 98   9  32  15
 99  12  97  78
 
 [100 rows x 3 columns])

### `data1.append(data2)` does not exist any more

That was the error in the earlier version of this notebook:

```
AttributeError: 'DataFrame' object has no attribute 'append'
```

Nothing was wrong with the idea. `DataFrame.append()` was deprecated in pandas
1.4 and **removed in pandas 2.0**. This environment runs pandas 3.0, so the
method is simply gone. Tutorials written before 2022 still use it.

The replacement is `pd.concat()`, which is a top-level function, not a method:
you pass it a **list** of frames.

In [37]:
pd.concat([data1,data2])

,a,b,c,d,x,y,z
0,39.0,47.0,19.0,15.0,NaN,NaN,NaN
1,28.0,24.0,72.0,9.0,NaN,NaN,NaN
2,47.0,33.0,62.0,97.0,NaN,NaN,NaN
3,19.0,67.0,74.0,40.0,NaN,NaN,NaN
4,47.0,65.0,29.0,50.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...
95,NaN,NaN,NaN,NaN,52.0,9.0,54.0
96,NaN,NaN,NaN,NaN,1.0,44.0,99.0
97,NaN,NaN,NaN,NaN,75.0,98.0,80.0
98,NaN,NaN,NaN,NaN,9.0,32.0,15.0


In [38]:
pd.concat([data1,data2]).shape

(200, 7)

Read the shape: **200 rows, 7 columns**.

`concat` stacks rows on top of each other by default (`axis='index'`). These
two frames share no column names, so pandas keeps all seven columns and fills
the gaps with `NaN` — `data1`'s rows have nothing to put under `x y z`, and
`data2`'s rows have nothing to put under `a b c d`.

Also look at the index down the left: it runs 0–99 and then 0–99 again. Each
frame kept its own labels, so the labels are now duplicated.

`ignore_index=True` throws the old labels away and numbers the result 0–199.

In [39]:
pd.concat([data1,data2],ignore_index=True).tail()

,a,b,c,d,x,y,z
195,NaN,NaN,NaN,NaN,52.0,9.0,54.0
196,NaN,NaN,NaN,NaN,1.0,44.0,99.0
197,NaN,NaN,NaN,NaN,75.0,98.0,80.0
198,NaN,NaN,NaN,NaN,9.0,32.0,15.0
199,NaN,NaN,NaN,NaN,12.0,97.0,78.0


In [40]:
pd.concat([data1,data2],axis='columns')

,a,b,c,d,x,y,z
0,39,47,19,15,36,54,26
1,28,24,72,9,75,72,40
2,47,33,62,97,43,22,20
3,19,67,74,40,8,29,40
4,47,65,29,50,90,30,81
...,...,...,...,...,...,...,...
95,33,59,43,79,52,9,54
96,12,40,20,94,1,44,99
97,38,3,28,56,75,98,80
98,90,94,13,77,9,32,15


Switching to `axis='columns'` glues the frames **side by side** instead: 100
rows, 7 columns, no `NaN` at all. That is what we actually wanted for two
frames with different columns.

The short version:

| Goal | Call |
|---|---|
| more rows, same columns | `pd.concat([a, b])` |
| more columns, same rows | `pd.concat([a, b], axis='columns')` |
| renumber the index afterwards | add `ignore_index=True` |

Side-by-side concat matches on the index. It does not look at any key column —
that job belongs to `merge`, next.

## 10. Merge — a SQL join

### Why `data3.merge(data4, on='a')` failed

That was the second error:

```
KeyError: 'a'
```

`on='a'` tells pandas "join these two frames using column `a`, which exists in
both". But `data4` had been built with the columns `x y z` — there was no `a`
in it to join on, so pandas raised `KeyError` on the missing key.

A merge needs a **shared key column**. So build the frames with one.

In [41]:
data3 = pd.DataFrame({
    'id':   [1, 2, 3, 4, 5],
    'name': ['Asha', 'Ravi', 'Meera', 'Kiran', 'Dev'],
})

data4 = pd.DataFrame({
    'id':   [2, 3, 5, 6],
    'city': ['Pune', 'Mumbai', 'Surat', 'Delhi'],
})

data3, data4

(   id   name
 0   1   Asha
 1   2   Ravi
 2   3  Meera
 3   4  Kiran
 4   5    Dev,
    id    city
 0   2    Pune
 1   3  Mumbai
 2   5   Surat
 3   6   Delhi)

In [42]:
data3.merge(data4,on='id')

,id,name,city
0,2,Ravi,Pune
1,3,Meera,Mumbai
2,5,Dev,Surat


Now `on='id'` works, because both frames have an `id` column.

Only ids 2, 3 and 5 are in **both** frames, so only those three rows come back.
Id 1 and 4 (no city) and id 6 (no name) are dropped.

That is an **inner join** — `how='inner'` is the default, exactly like SQL.

In [43]:
data3.merge(data4,on='id',how='left')

,id,name,city
0,1,Asha,NaN
1,2,Ravi,Pune
2,3,Meera,Mumbai
3,4,Kiran,NaN
4,5,Dev,Surat


In [44]:
data3.merge(data4,on='id',how='outer')

,id,name,city
0,1,Asha,NaN
1,2,Ravi,Pune
2,3,Meera,Mumbai
3,4,Kiran,NaN
4,5,Dev,Surat
5,6,NaN,Delhi


Same two frames, different `how`:

| `how` | Keeps | Result above |
|---|---|---|
| `'inner'` (default) | only ids in both | 3 rows |
| `'left'` | all of `data3` | 5 rows — ids 1 and 4 get `NaN` city |
| `'right'` | all of `data4` | 4 rows |
| `'outer'` | everything from both | 6 rows — ids 1, 4, 6 partly `NaN` |

These are the same four joins you already know from SQL, with the same
meanings.

Two practical notes:

- If the key has a different name in each frame, use
  `left_on='id', right_on='customer_id'` instead of `on=`.
- If the key is not unique, rows multiply — 3 matching rows on the left and 2
  on the right give 6 rows out. Check `.shape` after every merge. This is what
  made the original random-integer version of this cell a bad example even
  before the `KeyError`: random values from 1–99 across 100 rows repeat a lot,
  so the join would have exploded into thousands of meaningless rows.

## Recap

| Task | Call |
|---|---|
| one labelled column | `pd.Series(data, index=...)` |
| a table | `pd.DataFrame({...})` |
| load a file | `pd.read_csv(path)` |
| first look | `.info()`, `.head()`, `.tail()` |
| count categories | `.value_counts()` |
| rename a column | `df = df.rename(columns={'old':'new'})` |
| one column (Series) | `df['col']` |
| several columns (DataFrame) | `df[['a','b']]` |
| rows by label / position | `.loc[]` / `.iloc[]` |
| filter rows | `df[df['col'] > x]`, join with `&` and `|` |
| find holes | `.isnull()`, `.notnull()`, `.isnull().sum()` |
| fill holes | `df['c'] = df['c'].fillna(value)` |
| remove a column | `df.drop(['b'], axis='columns')` |
| stack frames | `pd.concat([a, b])` — `append` is gone |
| join on a key | `a.merge(b, on='id', how='left')` |

**The one habit to keep:** pandas methods return a copy. Assign the result back
or the change never happened.